# Ex9: CNN AlexNet CIFAR-10

In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as dataset
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
train_dataset = dataset.CIFAR10('../DATA',train=True,transform=transforms.ToTensor(),download=True)
test_dataset = dataset.CIFAR10('../DATA',train=False,transform=transforms.ToTensor(),download=True)

In [ ]:
plt.imshow(train_dataset.data[0])
plt.title(train_dataset.classes[train_dataset.targets[0]])

In [ ]:
print(f'X Train shape {train_dataset.data.shape} y train shape {np.array(train_dataset.targets).shape}')
print(f'X Train shape {test_dataset.data.shape} y train shape {np.array(test_dataset.targets).shape}')


In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=64,shuffle=False)

## AlexNet class (sort of)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self,in_channel=3,classses = 10):
        super().__init__()
        self.c1 = nn.Conv2d(in_channels=in_channel,out_channels=96,kernel_size=3,stride=1,padding=1) # (32x32x3) -> (32-3-2+1)=32x32x96
        self.p1 = nn.MaxPool2d(kernel_size=2,stride=2) #(32-2)/2 +1 = 16x16x96
        self.c2 = nn.Conv2d(in_channels=96,out_channels=192,kernel_size=3,stride=1,padding=1) # (16-3+2+1) = 16x16x192
        self.p2 = nn.MaxPool2d(kernel_size=2,stride=2) # 8x8x192
        self.c3 = nn.Conv2d(in_channels=192,out_channels=384,kernel_size=3,padding=1) # 8x8x384
        self.p3 = nn.MaxPool2d(kernel_size=2,stride=2)
        self.c4 = nn.Conv2d(in_channels=384,out_channels=256,kernel_size=3,padding=1) # 4x4x256
        self.c5 = nn.Conv2d(in_channels=256,out_channels=256,kernel_size=3,padding=1) # 4x4x256
        self.p4 = nn.MaxPool2d(kernel_size=2,stride=2)# (4-2)/2+1= 2x2x256
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(2*2*256,512)
        self.drp1 = nn.Dropout()
        self.fc2 = nn.Linear(512,256)
        self.drp2 = nn.Dropout()
        self.fc3 = nn.Linear(256,classses)

    def forward(self,x): 
        x = self.c1(x)
        x = F.relu(x)
        x = self.p1(x)

        x = self.c2(x)
        x = F.relu(x)
        x = self.p2(x)
        x = self.c3(x)
        x = F.relu(x)
        x = self.p3(x)
        x = self.c4(x)
        x= F.relu(x)
        x = self.c5(x)
        x= F.relu(x)
        x = self.p4(x)
        x = self.flat(x)
        x = self.fc1(x)
        x= F.relu(x)
        x = self.drp1(x)
        x = self.fc2(x)
        x= F.relu(x)
        x = self.drp2(x)
        return self.fc3(x)




torch.Size([128, 96, 32, 32])
torch.Size([128, 96, 16, 16])
torch.Size([128, 192, 16, 16])
torch.Size([128, 192, 8, 8])
torch.Size([128, 384, 8, 8])
torch.Size([128, 256, 8, 8])
torch.Size([128, 256, 8, 8])
torch.Size([128, 256, 4, 4])
torch.Size([128, 4096])
torch.Size([128, 512])
torch.Size([128, 256])

In [ ]:
model = AlexNet(3,10)
print(model)

## training

In [ ]:
epochs = 100
train_losses =[]
test_losses = []
train_acc =[]
test_acc = []


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device=device)
criterion = nn.CrossEntropyLoss()
criterion.to(device=device)
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

In [ ]:
def single_epoch(loader: torch.utils.data.DataLoader, epoch: int ,is_training = False):
    tmp_loss = []
    c_total = 0
    c_current=0

    for X,y in loader:
        if is_training:
            optimizer.zero_grad()
        
        X = X.to(device)
        y = y.to(device)

        y_hat = model(X)

        loss = criterion(y_hat,y)

        if is_training:
            loss.backward()
            optimizer.step()

        tmp_loss.append(loss.item())
        c_total += X.shape[0]
        
        _,argmax = torch.max(y_hat,1)

        c_current += (argmax == y).sum().item()
    
    acc = c_current / c_total
    return acc, np.mean(tmp_loss)

In [ ]:
from datetime import datetime as dt

starttime = dt.now()
for epoch in range(1,epochs+1):
   
   acc,loss = single_epoch(train_loader,epoch,True)
   train_losses.append(loss)
   train_acc.append(acc)
   acc,loss = single_epoch(test_loader,epoch,False)
   test_acc.append(acc)
   test_losses.append(loss.item())
   print(f'{dt.now()-starttime}:: {epoch}/{epochs}train loss {train_losses[-1]} train acc {train_acc[-1]} test loss {test_losses[-1]} test acc {test_acc[-1]}')

In [ ]:
its = [x for x in range(epochs)]
plt.plot(train_losses,label='train loss',color='orange')
plt.plot(test_losses,label='test loss',color='red')
plt.legend()

In [ ]:
plt.plot(train_acc,label='train_acc',color='blue')
plt.plot(test_acc,label='test_acc',color='green')
plt.legend()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:

Y_hats = []
Y = []

for X,y in test_loader:
    X,y = X.to(device),y.to(device)
    y_hat = model(X)
    _,y_hat = torch.max(y_hat,1)

    with torch.no_grad():
        Y = np.concatenate((Y,y.cpu().numpy()))
        Y_hats = np.concatenate((Y_hats,y_hat.cpu().numpy()))

In [ ]:
ConfusionMatrixDisplay.from_predictions(Y,Y_hats,display_labels=test_dataset.classes)

In [ ]:
idxs = np.where(Y != Y_hats)[0]

In [ ]:
i = np.random.choice(idxs)
plt.imshow(test_dataset.data[i])
t,f = int(Y[i]),int(Y_hats[i])
print(t,f)
plt.title(f'{test_dataset.classes[t]} as {test_dataset.classes[f]}')